In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import copy
import json

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv("heart.csv")
print(df.head())
print(df["target"].value_counts())
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.savefig("outputs/figures/corr_matrix.png")
plt.close()

In [ ]:
from src.data import load_and_preprocess
X_tr, X_val, X_te, y_tr, y_val, y_te = load_and_preprocess("heart.csv")
input_dim = X_tr.shape[1]

num_clients = 5
client_data_indices = np.array_split(np.arange(len(X_tr)), num_clients)

client_dataloaders = []
for i in range(num_clients):
    idx = client_data_indices[i]
    cx = torch.tensor(X_tr[idx], dtype=torch.float32)
    cy = torch.tensor(y_tr[idx], dtype=torch.float32)
    ds = TensorDataset(cx, cy)
    dl = DataLoader(ds, batch_size=16, shuffle=True)
    client_dataloaders.append(dl)

val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False)

test_ds = TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.float32))
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
from src.model import HeartNet
from src.client import FLClient
from src.server import fedavg
from src.metrics import eval_model

global_model = HeartNet(input_dim).to(device)
global_weights = global_model.state_dict()

clients = []
for i in range(num_clients):
    c = FLClient(client_id=i, dataloader=client_dataloaders[i], model=global_model, lr=0.05, epochs=3, device=device)
    clients.append(c)

In [ ]:
num_rounds = 30
val_accs = []

for r in range(num_rounds):
    client_weights = []
    
    for c in clients:
        w = c.train(global_weights)
        client_weights.append(w)
        
    global_weights = fedavg(client_weights)
    global_model.load_state_dict(global_weights)
    
    vacc, vprec, vrec, vf1, _, _ = eval_model(global_model, val_dl, device)
    val_accs.append(vacc)
    
    if (r+1) % 5 == 0:
        print(f"Round {r+1} - Val Acc: {vacc:.4f}, F1: {vf1:.4f}")
        torch.save(global_weights, f"outputs/checkpoints/model_round_{r+1}.pth")

with open("outputs/logs/val_accs.json", "w") as f:
    json.dump(val_accs, f)

In [ ]:
global_model.load_state_dict(torch.load(f"outputs/checkpoints/model_round_{num_rounds}.pth"))
tacc, tprec, trec, tf1, test_preds, test_labels = eval_model(global_model, test_dl, device)

print(f"Test Accuracy: {tacc:.4f}")
print(f"Test Precision: {tprec:.4f}")
print(f"Test Recall: {trec:.4f}")
print(f"Test F1: {tf1:.4f}")

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.savefig("outputs/figures/confusion_matrix.png")
plt.close()

In [ ]:
plt.figure()
plt.plot(range(1, num_rounds+1), val_accs, marker='o', label="Validation Accuracy")
plt.xlabel("Communication Round")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("outputs/figures/training_curve.png")
plt.close()

In [ ]:
global_model.eval()
raw_preds = []
with torch.no_grad():
    for x, _ in test_dl:
        x = x.to(device)
        raw_preds.extend(global_model(x).squeeze().cpu().numpy())

fpr, tpr, _ = roc_curve(test_labels, raw_preds)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.savefig("outputs/figures/roc_curve.png")
plt.close()